# Week 24: Final independence assessment

**This is the exam that matters.** It tests whether you can build, debug, evaluate and explain AI systems **without help**. Passing it means you've reached the goal you set in October.

### Rules
- **No AI tools. No hints** (there are none in this notebook). **No copying from your earlier notebooks or the capstone.**
- Allowed: official documentation (docs.python.org, numpy.org, scikit-learn.org, pytorch.org) and a blank sheet of paper.
- Do each part in **one sitting**, and record the time you started and finished.
- If you're stuck for 20 minutes, write down what you tried and move on. That's information, not failure.

| Part | Task | Time limit |
|---|---|---|
| 1 | An unfamiliar problem: a tested package, built from scratch | 3 h |
| 2 | Debugging: a broken pipeline with hidden problems | 45 min |
| 3 | Derivations on paper | 45 min |
| 4 | Reproducibility: a fresh clone of your capstone | 30 min |
| 5 | Explain it: a recorded oral explanation | 30 min |
| 6 | Final competency matrix with evidence | 1 h |

In [ ]:
import numpy as np
from pathlib import Path
WORK = Path("../work/week24"); WORK.mkdir(parents=True, exist_ok=True)

---
# Part 1: An unfamiliar problem (3 h)

**The task is regression**, which you haven't done since Week 6. scikit-learn's bundled `load_diabetes()` dataset has 442 patients, 10 baseline variables (age, sex, BMI, blood pressure, six blood serum measurements; already standardised), and a measure of disease progression one year later as the target.

Build a package in `../work/week24/challenge/`:
```
challenge/
├── ridge/__init__.py
├── ridge/model.py        # class Ridge: fit(X, y) / predict(X); alpha; closed form AND gradient descent (method="closed" | "gd")
│                         # the intercept is NOT penalised (centre X and y, solve, then recover the intercept)
├── ridge/cv.py           # kfold_indices(n, k, seed); cross_val_mse(model_factory, X, y, k, seed) -> per-fold MSE array
└── tests/test_ridge.py   # ≥ 6 tests: matches sklearn.linear_model.Ridge; GD ≈ closed form; alpha=0 = OLS;
                          # large alpha shrinks coefficients towards 0; folds partition the indices; shapes
```
Then, in this notebook:
1. Choose `alpha` from {0.01, 0.1, 1, 10, 100} by 10-fold CV on a **training split** (80%, seed 0).
2. Compare your tuned Ridge with `HistGradientBoostingRegressor(random_state=0)` using the **same** 10 folds on the training split. Report the per-fold MSE for each, the mean difference with a paired test across folds (from scratch), and the result on the held-out 20%.
3. In `CHALLENGE.md`: write the results in 5 sentences, and explain why the simpler model might win on 442 samples.

In [ ]:
import subprocess, sys, importlib
sys.path.insert(0, str((WORK / "challenge").resolve()))
# After building the package in VS Code, run the checks below.

In [ ]:
importlib.invalidate_caches()
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=WORK / "challenge", capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0 and (WORK / "challenge/tests/test_ridge.py").read_text().count("def test_") >= 6
from ridge.model import Ridge
from ridge.cv import cross_val_mse
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge as SkRidge
X, y = load_diabetes(return_X_y=True)
ours = Ridge(alpha=1.0).fit(X, y); ref = SkRidge(alpha=1.0).fit(X, y)
assert np.allclose(ours.predict(X), ref.predict(X), atol=1e-6)
gd = Ridge(alpha=1.0, method="gd").fit(X, y)
assert np.mean((gd.predict(X) - ref.predict(X)) ** 2) < 1.0
assert len(cross_val_mse(lambda: Ridge(alpha=1.0), X, y, k=10, seed=0)) == 10
print("✅ Part 1 package checks passed. Now do the comparison and write CHALLENGE.md.")

In [ ]:
# Part 1, steps 1–2: your analysis here

---
# Part 2: Debugging (45 min)

This function claims to estimate how well a classifier generalises on the breast cancer data. It has **several problems**: some make the estimate optimistic, some are plain bugs. Find and fix them all, and list each one in the markdown cell below with its effect. *(How many there are is part of the test.)*

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def estimate_generalisation(seed=0):
    X, y = load_breast_cancer(return_X_y=True)
    X = StandardScaler().fit_transform(X)
    X = SelectKBest(f_classif, k=10).fit_transform(X, y)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25)
    best_acc, best_C = 0, None
    for C in [0.01, 0.1, 1, 10, 100]:
        acc = LogisticRegression(C=C, max_iter=5000).fit(X_tr, y_tr).score(X_te, y_te)
        if acc > best_acc:
            best_acc, best_C = acc, C
    return best_acc

In [ ]:
import inspect
src = inspect.getsource(estimate_generalisation)
a1, a2 = estimate_generalisation(0), estimate_generalisation(0)
assert a1 == a2, "still not reproducible"
assert "Pipeline" in src or "make_pipeline" in src, "preprocessing must be fitted inside the training data only"
assert "cross_val" in src or "GridSearchCV" in src or "validation" in src.lower(), "C must be chosen without the test set"
print("✅ Part 2 structural checks passed. Accuracy:", round(a1, 3))

✍️ *Problem 1: …, effect: … (list them all)* (double-click to edit)



---
# Part 3: Derivations on paper (45 min)

Photograph your pages into `../work/week24/derivations/`. No notes.
1. The closed-form Ridge solution $w = (X^\top X + \alpha I)^{-1}X^\top y$, from the loss, **and** its interpretation as MAP with a Gaussian prior.
2. The gradient of binary cross-entropy for logistic regression.
3. The backward pass of a 2-layer ReLU MLP with softmax + cross-entropy (all four gradients).
4. Eigenvalues of $\begin{pmatrix} 3 & 1 \\ 1 & 3 \end{pmatrix}$, plus the eigenvectors, with a geometric explanation.
5. The BM25 formula, with an explanation of each part, and nDCG@3 for the ranking [0, 2, 1] with grades {doc2: 2, doc3: 1}.
6. Bayes: 2% prevalence, 95% sensitivity, 90% specificity. Compute the PPV.

---
# Part 4: Reproducibility (30 min)

In a **new terminal**:
```bash
rm -rf /tmp/fresh && git clone https://github.com/<you>/medsearch-lab.git /tmp/fresh && cd /tmp/fresh
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements-lock.txt && pip install -e .
SKIP_GPU=1 bash scripts/reproduce.sh
git status        # nothing should have changed except regenerated, identical results
diff results/table_A.md <(git show HEAD:results/table_A.md) && echo "Table A reproduced exactly"
```
Write down the runtime and any problem you hit. A non-reproducible table is a bug: fix it and push.

✍️ *Runtime, problems, and fixes: …* (double-click to edit)



---
# Part 5: Explain it (30 min, recorded)

Record yourself (phone or screen recording) explaining **without slides**, about 3 minutes each:
1. How your search engine ranks documents, from tokenisation to nDCG.
2. How you'd know whether a new model is truly better than the old one.
3. Why your imaging model must not be used clinically, and what it would take to change that.

Watch the recording back. Note each place where you hesitated, and revise those topics.

✍️ *Links to the recordings, plus what I hesitated on: …* (double-click to edit)



---
# Part 6: The final competency matrix (1 h)

Update `COMPETENCY_TRACKER.md` one last time. For each skill, give the **level (0–4)** and **evidence links**. A level 4 needs evidence from *this* assessment or the capstone, done independently.

Then write `../FINAL_REFLECTION.md`:
- your three strongest skills, with evidence;
- your three weakest, and a concrete 4-week plan for each;
- what you'd tell yourself in October;
- your next step: a master's application (deadline **1 June**), a side job, or both.

**Send me** the tracker, `CHALLENGE.md`, the Part 2 fixes and the reflection. I'll verify every level-4 claim against its evidence, the way a strict examiner would, and give you a final written assessment.